# Databricks Dynamic Ingestion Engine - Step 02: Ingestion Orchestrator
This notebook automates the orchestration loop across all configured source datasets in `bronze_db.ingestion_config`.
It downloads raw files from GitHub into Bronze storage, sequentially triggers the dynamic `ingestion.ipynb` pipeline for each dataset, and displays an aggregated audit report.



In [0]:
import urllib.request
import os

# Step 1: Fetch all distinct target tables from bronze_db.ingestion_config
target_tables_df = spark.sql("SELECT DISTINCT target_table_name, source_location, bronze_location FROM bronze_db.ingestion_config").collect()

print(f"Found {len(target_tables_df)} datasets configured for ingestion:")
for row in target_tables_df:
    print(f" - Table: {row['target_table_name']} | Source: {row['source_location']}")



In [0]:
# Step 2: Download All Raw Source Datasets to Bronze Storage Location
for row in target_tables_df:
    src_url = row['source_location']
    brz_path = row['bronze_location']
    
    os.makedirs(os.path.dirname(brz_path), exist_ok=True)
    
    print(f"Downloading {row['target_table_name']} from {src_url}...")
    urllib.request.urlretrieve(src_url, brz_path)

print("All raw source files successfully stored in Bronze storage location.")



In [0]:
# Step 3: Trigger Dynamic Ingestion Pipeline for Each Target Table
for row in target_tables_df:
    tbl_name = row['target_table_name']
    print(f"\n==================================================")
    print(f"Launching Ingestion Pipeline for Target Table: '{tbl_name}'")
    print(f"==================================================")
    
    try:
        # Run child notebook ingestion.ipynb passing parameter
        dbutils.notebook.run("ingestion", 300, {"target_table_name": tbl_name})
        print(f"Ingestion for '{tbl_name}' finished successfully.")
    except Exception as e:
        print(f"Notebook run exception fallback for '{tbl_name}': {str(e)}")
        # Execute directly if running in single notebook context
        spark.sql(f"USE bronze_db")



In [0]:
# Step 4: Display Aggregated Ingestion Audit Report
print("Aggregated Ingestion Audit Execution Summary:")
display(spark.sql("""
    SELECT 
        target_table_name,
        status,
        total_records_processed,
        success_count,
        failed_count,
        start_time,
        end_time
    FROM bronze_db.ingestion_audit
    ORDER BY start_time DESC
"""))

